## 1. Import Libraries and Setup


In [2]:
import pandas as pd
import os

In [3]:
# Configuration
RAW_DATA_DIR = '../../data/raw'
PREPROCESS_DIR = '../../data/preprocessed'

# Ensure output directory exists
os.makedirs(PREPROCESS_DIR, exist_ok=True)

print("=" * 80)
print("OULAD Dataset Preprocessing Pipeline")
print("=" * 80)
print(f"\nRaw data location: {RAW_DATA_DIR}")
print(f"Output location: {PREPROCESS_DIR}\n")

OULAD Dataset Preprocessing Pipeline

Raw data location: ../../data/raw
Output location: ../../data/preprocessed



## 2. Utility Functions

Define helper functions for common preprocessing tasks.


### 2.1 Downcast Integers


In [4]:
def downcast_integers(df):
    """
    Downcast all integer columns to the smallest possible type.
    Reduces memory usage significantly.
    
    Mapping logic:
    - Positive integers: uint8/uint16/uint32 based on max value
    - Negative integers: int8/int16/int32 based on min/max range
    """
    int_cols = df.select_dtypes(include=['int64', 'int32']).columns
    for col in int_cols:
        min_val = df[col].min()
        max_val = df[col].max()
        
        if min_val >= 0:
            # Non-negative: use unsigned types
            if max_val < 256:
                df[col] = df[col].astype('uint8')
            elif max_val < 65536:
                df[col] = df[col].astype('uint16')
            else:
                df[col] = df[col].astype('uint32')
        else:
            # Can be negative: use signed types
            if min_val > -128 and max_val < 127:
                df[col] = df[col].astype('int8')
            elif min_val > -32768 and max_val < 32767:
                df[col] = df[col].astype('int16')
            else:
                df[col] = df[col].astype('int32')
    
    return df

### 2.2 Downcast Floats


In [5]:
def downcast_floats(df):
    """
    Downcast all float columns to float32 for memory optimization.
    float32 provides good precision while using 50% less memory than float64.
    
    Mapping logic:
    - float64 -> float32 (4 bytes vs 8 bytes per value)
    """
    float_cols = df.select_dtypes(include=['float64']).columns
    for col in float_cols:
        df[col] = df[col].astype('float32')
    
    if len(float_cols) > 0:
        print(f"  Downcast {len(float_cols)} float64 column(s) to float32")
    
    return df


### 2.3 Convert Code Columns


In [6]:
def convert_code_columns(df):
    """
    Convert code_module and code_presentation to category dtype.
    Ensures consistent data types across all files for seamless joining.
    """
    if 'code_module' in df.columns:
        df['code_module'] = df['code_module'].astype('category')
    if 'code_presentation' in df.columns:
        df['code_presentation'] = df['code_presentation'].astype('category')
    if 'assessment_type' in df.columns:
        df['assessment_type'] = df['assessment_type'].astype('category')
    return df

### 2.4 Save Processed File


In [7]:
def save_processed_file(df, filename, description):
    """
    Save processed dataframe and print summary statistics.
    
    Parameters:
    - df: Processed dataframe
    - filename: Output filename
    - description: Brief description of processing applied
    """
    output_path = os.path.join(PREPROCESS_DIR, filename)
    df.to_csv(output_path, index=False)
    
    memory_usage = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"  ✓ Saved: {filename}")
    print(f"    Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"    Memory: {memory_usage:.2f} MB")
    print(f"    {description}\n")

## 3. File Processing


### 3.1 Assessments.csv

**Processing**: Dtype optimization + category conversion (no cleaning yet)

This file will be cleaned post-merge with student assessment scores.


In [8]:
print("\n[1/7] Processing assessments.csv...")
print("-" * 80)

assessments = pd.read_csv(os.path.join(RAW_DATA_DIR, 'assessments.csv'))

# Optimize dtypes
assessments = downcast_integers(assessments)
assessments = downcast_floats(assessments)
assessments = convert_code_columns(assessments)

save_processed_file(assessments, 'assessments.csv', 
                   "Dtype optimization + category conversion (no cleaning yet)")

# Data type after processing
print(assessments.dtypes)


[1/7] Processing assessments.csv...
--------------------------------------------------------------------------------
  Downcast 2 float64 column(s) to float32
  ✓ Saved: assessments.csv
    Shape: 206 rows × 6 columns
    Memory: 0.00 MB
    Dtype optimization + category conversion (no cleaning yet)

code_module          category
code_presentation    category
id_assessment          uint16
assessment_type      category
date                  float32
weight                float32
dtype: object


### 3.2 StudentRegistration.csv

**Processing**: Dtype optimization + category conversion (no cleaning yet)

This file will be cleaned post-merge with student information.


In [9]:
print("[2/7] Processing studentRegistration.csv...")
print("-" * 80)

student_reg = pd.read_csv(os.path.join(RAW_DATA_DIR, 'studentRegistration.csv'))

# Optimize dtypes
student_reg = downcast_integers(student_reg)
student_reg = downcast_floats(student_reg)
student_reg = convert_code_columns(student_reg)

save_processed_file(student_reg, 'studentRegistration.csv',
                   "Dtype optimization + category conversion (no cleaning yet)")

print(f"Data Types after processing studentRegistration.csv:\n{student_reg.dtypes}")

[2/7] Processing studentRegistration.csv...
--------------------------------------------------------------------------------
  Downcast 2 float64 column(s) to float32
  ✓ Saved: studentRegistration.csv
    Shape: 32,593 rows × 5 columns
    Memory: 0.44 MB
    Dtype optimization + category conversion (no cleaning yet)

Data Types after processing studentRegistration.csv:
code_module            category
code_presentation      category
id_student               uint32
date_registration       float32
date_unregistration     float32
dtype: object


### 3.3 StudentAssessment.csv

**Processing**: Dtype optimization + category conversion (no cleaning yet)

This file contains student assessment scores and will be cleaned post-merge.


In [10]:
print("[3/7] Processing studentAssessment.csv...")
print("-" * 80)

student_assess = pd.read_csv(os.path.join(RAW_DATA_DIR, 'studentAssessment.csv'))

# Optimize dtypes
student_assess = downcast_integers(student_assess)
student_assess = downcast_floats(student_assess)
student_assess = convert_code_columns(student_assess)

save_processed_file(student_assess, 'studentAssessment.csv',
                   "Dtype optimization + category conversion (no cleaning yet)")

print(f"Data Types after processing studentAssessment.csv:\n{student_assess.dtypes}")

[3/7] Processing studentAssessment.csv...
--------------------------------------------------------------------------------
  Downcast 1 float64 column(s) to float32
  ✓ Saved: studentAssessment.csv
    Shape: 173,912 rows × 5 columns
    Memory: 2.16 MB
    Dtype optimization + category conversion (no cleaning yet)

Data Types after processing studentAssessment.csv:
id_assessment      uint16
id_student         uint32
date_submitted      int16
is_banked           uint8
score             float32
dtype: object


### 3.4 Courses.csv

**Processing**:

1. Convert code_module and code_presentation to category
2. Downcast module_presentation_length to int16
3. Optimize other integer columns


In [11]:
print("[4/7] Processing courses.csv...")
print("-" * 80)

courses = pd.read_csv(os.path.join(RAW_DATA_DIR, 'courses.csv'))

print(f"  Initial shape: {courses.shape}")

# Convert code columns to category
courses = convert_code_columns(courses)

# Downcast module_presentation_length to int16
if 'module_presentation_length' in courses.columns:
    courses['module_presentation_length'] = courses['module_presentation_length'].astype('int16')

# Optimize other integers and floats
courses = downcast_integers(courses)
courses = downcast_floats(courses)

save_processed_file(courses, 'courses.csv',
                   "Category conversion + int16 downcast for length")

print(f"Data Types after processing courses.csv:\n{courses.dtypes}")

[4/7] Processing courses.csv...
--------------------------------------------------------------------------------
  Initial shape: (22, 3)
  ✓ Saved: courses.csv
    Shape: 22 rows × 3 columns
    Memory: 0.00 MB
    Category conversion + int16 downcast for length

Data Types after processing courses.csv:
code_module                   category
code_presentation             category
module_presentation_length       int16
dtype: object


### 3.5 VLE.csv

**Processing**:

1. Drop week_from and week_to columns (>80% missing values)
2. Convert activity_type, code_module, code_presentation to category
3. Optimize integer columns


In [12]:
print("[5/7] Processing vle.csv...")
print("-" * 80)

vle = pd.read_csv(os.path.join(RAW_DATA_DIR, 'vle.csv'))

print(f"  Initial shape: {vle.shape}")

# Drop week_from and week_to (>80% missing values)
missing_week_from = vle['week_from'].isnull().sum() / len(vle) * 100
missing_week_to = vle['week_to'].isnull().sum() / len(vle) * 100
print(f"  week_from missing: {missing_week_from:.1f}% → Dropping")
print(f"  week_to missing: {missing_week_to:.1f}% → Dropping")

vle = vle.drop(columns=['week_from', 'week_to'])

# Convert activity_type, code_module, code_presentation to category
vle['activity_type'] = vle['activity_type'].astype('category')
vle = convert_code_columns(vle)

# Optimize integers and floats
vle = downcast_integers(vle)
vle = downcast_floats(vle)

# Add new feature: total_unique_activities
vle = vle.groupby(
    ['code_module', 'code_presentation']
    ).agg({
        'id_site': 'nunique'
    }).reset_index().rename(columns={'id_site': 'total_unique_activities'})

save_processed_file(vle, 'vle.csv',
                   "Dropped week_from/week_to + category conversion")

print(f"Data Types after processing vle.csv:\n{vle.dtypes}")

[5/7] Processing vle.csv...
--------------------------------------------------------------------------------
  Initial shape: (6364, 6)
  week_from missing: 82.4% → Dropping
  week_to missing: 82.4% → Dropping
  ✓ Saved: vle.csv
    Shape: 22 rows × 3 columns
    Memory: 0.00 MB
    Dropped week_from/week_to + category conversion

Data Types after processing vle.csv:
code_module                category
code_presentation          category
total_unique_activities       int64
dtype: object


### 3.6 StudentInfo.csv

**Processing**:

1. Impute imd_band using region mode
2. Convert code_module and code_presentation to category
3. Convert ordinal columns (age_band, highest_education, imd_band)
4. Convert nominal columns (gender, region, disability, final_result)
5. Optimize integer columns


In [13]:
print("[6/7] Processing studentInfo.csv...")
print("-" * 80)

student_info = pd.read_csv(os.path.join(RAW_DATA_DIR, 'studentInfo.csv'))

print(f"  Initial shape: {student_info.shape}")

[6/7] Processing studentInfo.csv...
--------------------------------------------------------------------------------
  Initial shape: (32593, 12)


#### 3.6.1 Impute IMD Band by Region Mode


Identify Missing Data by Region


In [14]:
# Create a summary of missing IMD values by region
missing_by_region = student_info[student_info['imd_band'].isnull()]['region'].value_counts()

print("Count of missing 'imd_band' values per region:")
if missing_by_region.empty:
    print("No missing values found.")
else:
    print(missing_by_region)

# To see it as a percentage of that region's total students:
total_per_region = student_info['region'].value_counts()
pct_missing = (missing_by_region / total_per_region) * 100
print("\nPercentage of missing IMD values per region:")
print(pct_missing.dropna().sort_values(ascending=False))

Count of missing 'imd_band' values per region:
region
North Region            731
Ireland                 266
South Region             48
West Midlands Region     39
Scotland                 12
South West Region         5
North Western Region      5
Yorkshire Region          5
Name: count, dtype: int64

Percentage of missing IMD values per region:
region
North Region            40.098738
Ireland                 22.466216
South Region             1.552393
West Midlands Region     1.510457
Scotland                 0.348230
Yorkshire Region         0.249252
South West Region        0.205255
North Western Region     0.172058
Name: count, dtype: float64


In [15]:
print("\nStep 1: Imputing imd_band by region mode...")
missing_imd = student_info['imd_band'].isnull().sum()
print(f"    Missing imd_band values: {missing_imd}")

region_modes = {}

for region in student_info['region'].unique():
    # Calculate mode for this specific region
    mode_val = student_info[student_info['region'] == region]['imd_band'].mode()
    
    if not mode_val.empty:
        region_modes[region] = mode_val[0]
        # Apply imputation
        student_info.loc[
            (student_info['region'] == region) & (student_info['imd_band'].isnull()), 
            'imd_band'
        ] = mode_val[0]

# Print the full mapping clearly
print("\nRegion-to-IMD Mode Mapping:")
for reg, val in region_modes.items():
    print(f" - {reg:25} -> Mode: {val}")

remaining_missing = student_info['imd_band'].isnull().sum()
print(f"\nImputed: {missing_imd - remaining_missing} values")
print(f"Remaining missing: {remaining_missing}")


Step 1: Imputing imd_band by region mode...
    Missing imd_band values: 1111

Region-to-IMD Mode Mapping:
 - East Anglian Region       -> Mode: 90-100%
 - Scotland                  -> Mode: 50-60%
 - North Western Region      -> Mode: 0-10%
 - South East Region         -> Mode: 60-70%
 - West Midlands Region      -> Mode: 0-10%
 - Wales                     -> Mode: 20-30%
 - North Region              -> Mode: 10-20
 - South Region              -> Mode: 90-100%
 - Ireland                   -> Mode: 0-10%
 - South West Region         -> Mode: 30-40%
 - East Midlands Region      -> Mode: 10-20
 - Yorkshire Region          -> Mode: 0-10%
 - London Region             -> Mode: 10-20

Imputed: 1111 values
Remaining missing: 0


#### 3.6.2 Convert Code Columns to Category


In [16]:
print("\n  Step 2: Converting code columns to category...")
student_info = convert_code_columns(student_info)


  Step 2: Converting code columns to category...


#### 3.6.3 Convert Ordinal Columns with Specified Order


In [17]:
print("  Step 3: Converting ordinal columns with string matching...")

# --- 1. age_band ---
# Map the '55<=' from your data to our '55+' label
student_info['age_band'] = student_info['age_band'].replace({'55<=': '55+'})

age_band_order = ['0-35', '35-55', '55+']
student_info['age_band'] = pd.Categorical(
    student_info['age_band'],
    categories=age_band_order,
    ordered=True
)

print(f"   age_band: {len(age_band_order)} categories")

# --- 2. imd_band ---
# Ensure the values have the '%' if they are missing, and strip whitespace
student_info['imd_band'] = student_info['imd_band'].astype(str).str.strip()
# If your data looks like '10-20', this adds the '%' to match the order list
student_info.loc[~student_info['imd_band'].str.contains('%') & (student_info['imd_band'] != 'nan'), 'imd_band'] += '%'

imd_band_order = ['0-10%', '10-20%', '20-30%', '30-40%', '40-50%', 
                  '50-60%', '60-70%', '70-80%', '80-90%', '90-100%']

student_info['imd_band'] = pd.Categorical(
    student_info['imd_band'],
    categories=imd_band_order,
    ordered=True
)

print(f"   imd_band: {len(imd_band_order)} categories")

# --- 3. highest_education ---
education_order = ['No Formal quals', 'Lower Than A Level', 'A Level or Equivalent', 
                   'HE Qualification', 'Post Graduate Qualification']
student_info['highest_education'] = student_info['highest_education'].str.strip()
student_info['highest_education'] = pd.Categorical(
    student_info['highest_education'],
    categories=education_order,
    ordered=True
)

print(f"   highest_education: {len(education_order)} categories")

print(f"Missing Values Check:\n{student_info[['age_band', 'imd_band', 'highest_education']].isnull().sum()}")

  Step 3: Converting ordinal columns with string matching...
   age_band: 3 categories
   imd_band: 10 categories
   highest_education: 5 categories
Missing Values Check:
age_band             0
imd_band             0
highest_education    0
dtype: int64


#### 3.6.4 Convert Nominal Columns to Category


In [18]:
print("  Step 4: Converting nominal columns...")
nominal_cols = ['gender', 'region', 'disability', 'final_result']
for col in nominal_cols:
    if col in student_info.columns:
        student_info[col] = student_info[col].astype('category')
        print(f"    {col}: {student_info[col].nunique()} categories")

  Step 4: Converting nominal columns...
    gender: 2 categories
    region: 13 categories
    disability: 2 categories
    final_result: 4 categories


#### 3.6.5 Optimize Integer Columns and Save


In [19]:
student_info = downcast_integers(student_info)
student_info = downcast_floats(student_info)

save_processed_file(student_info, 'studentInfo.csv',
                   "IMD imputation + ordinal/nominal category conversion")

print(f"Data Types after processing studentInfo.csv:\n{student_info.dtypes}")

  ✓ Saved: studentInfo.csv
    Shape: 32,593 rows × 12 columns
    Memory: 0.50 MB
    IMD imputation + ordinal/nominal category conversion

Data Types after processing studentInfo.csv:
code_module             category
code_presentation       category
id_student                uint32
gender                  category
region                  category
highest_education       category
imd_band                category
age_band                category
num_of_prev_attempts       uint8
studied_credits           uint16
disability              category
final_result            category
dtype: object


### 3.7 StudentVle.csv

**Processing**:

1. Row-level cleaning: Group by module/presentation/student/site/date and sum sum_click
2. Feature engineering: Aggregate at (module, presentation, student) level to create:
   - total_clicks (sum of clicks)
   - avg_clicks_per_day (mean clicks)
3. Save as studentVle.csv


In [20]:
print("[7/7] Processing studentVle.csv...")
print("-" * 80)

student_vle = pd.read_csv(os.path.join(RAW_DATA_DIR, 'studentVle.csv'))

print(f"  Initial shape: {student_vle.shape}")

[7/7] Processing studentVle.csv...
--------------------------------------------------------------------------------
  Initial shape: (10655280, 6)


#### 3.7.1 Row-Level Cleaning (Group and Sum)


In [21]:
print("\n  Step 1: Row-level cleaning (grouping and summing sum_click)...")
student_vle_cleaned = student_vle.groupby(
    ['code_module', 'code_presentation', 'id_student', 'id_site', 'date'],
    as_index=False
)['sum_click'].sum()

print(f"    After grouping: {student_vle_cleaned.shape[0]:,} rows")
print(f"    Removed duplicates: {student_vle.shape[0] - student_vle_cleaned.shape[0]:,} rows")
print(f"    Preview:\n{student_vle_cleaned.head()}")


  Step 1: Row-level cleaning (grouping and summing sum_click)...
    After grouping: 8,459,320 rows
    Removed duplicates: 2,195,960 rows
    Preview:
  code_module code_presentation  id_student  id_site  date  sum_click
0         AAA             2013J       11391   546614    -5          7
1         AAA             2013J       11391   546614     0         10
2         AAA             2013J       11391   546614     1          9
3         AAA             2013J       11391   546614     2          3
4         AAA             2013J       11391   546614     6          1


#### 3.7.2 Feature Engineering (Aggregate at Student-Module-Presentation Level)


In [22]:
print("\n  Step 2: Feature engineering (aggregating at student-module-presentation level)...")

student_vle_agg = student_vle_cleaned.groupby(
    ['code_module', 'code_presentation', 'id_student']
).agg({
    'sum_click': ['sum', 'mean'],
    'id_site': ['count', 'nunique']  # Count of interactions
}).reset_index()

# Flatten column names
student_vle_agg.columns = ['code_module', 'code_presentation', 'id_student', 
                           'total_clicks', 'avg_clicks_per_day', 'num_sites', 'num_unique_sites']

print(f"    Aggregated shape: {student_vle_agg.shape}")
print(f"    Preview:\n{student_vle_agg.head()}")


  Step 2: Feature engineering (aggregating at student-module-presentation level)...
    Aggregated shape: (29228, 7)
    Preview:
  code_module code_presentation  id_student  total_clicks  avg_clicks_per_day  \
0         AAA             2013J       11391           934            5.592814   
1         AAA             2013J       28400          1435            4.019608   
2         AAA             2013J       30268           281            4.460317   
3         AAA             2013J       31604          2158            3.952381   
4         AAA             2013J       32885          1034            3.324759   

   num_sites  num_unique_sites  
0        167                55  
1        357                84  
2         63                22  
3        546                82  
4        311                66  


#### 3.7.3 Convert Columns and Optimize Dtypes


In [23]:
student_vle_agg = convert_code_columns(student_vle_agg)
student_vle_agg = downcast_integers(student_vle_agg)
student_vle_agg = downcast_floats(student_vle_agg)

save_processed_file(student_vle_agg, 'studentVle.csv',
                   "Row-level cleaning + feature engineering (total_clicks, avg_clicks_per_day)")

print(f"Data Types after processing studentVle.csv:\n{student_vle_agg.dtypes}")

  Downcast 1 float64 column(s) to float32
  ✓ Saved: studentVle.csv
    Shape: 29,228 rows × 7 columns
    Memory: 0.45 MB
    Row-level cleaning + feature engineering (total_clicks, avg_clicks_per_day)

Data Types after processing studentVle.csv:
code_module           category
code_presentation     category
id_student              uint32
total_clicks            uint16
avg_clicks_per_day     float32
num_sites               uint16
num_unique_sites        uint16
dtype: object


## 4. Summary and Final Report


In [24]:
print("\n" + "=" * 80)
print("PREPROCESSING COMPLETE")
print("=" * 80)

# List all processed files
preprocess_files = sorted([f for f in os.listdir(PREPROCESS_DIR) if f.endswith('.csv')])
print(f"\nProcessed files saved to '{PREPROCESS_DIR}/':\n")

total_size = 0
for i, filename in enumerate(preprocess_files, 1):
    filepath = os.path.join(PREPROCESS_DIR, filename)
    filesize = os.path.getsize(filepath) / 1024 / 1024
    total_size += filesize
    print(f"  {i}. {filename:<35} ({filesize:.2f} MB)")

print(f"\nTotal preprocessed data size: {total_size:.2f} MB")
print("\n✓ All files successfully preprocessed and ready for modeling!")


PREPROCESSING COMPLETE

Processed files saved to '../../data/preprocessed/':

  1. assessments.csv                     (0.01 MB)
  2. courses.csv                         (0.00 MB)
  3. studentAssessment.csv               (4.10 MB)
  4. studentInfo.csv                     (2.56 MB)
  5. studentRegistration.csv             (0.83 MB)
  6. studentVle.csv                      (1.07 MB)
  7. vle.csv                             (0.00 MB)

Total preprocessed data size: 8.57 MB

✓ All files successfully preprocessed and ready for modeling!
